In [0]:
# importing pyspark functions and types functions
from pyspark.sql import functions as F, types as T

rows_customers = [
    (1,  "Asha",  "IN", True),
    (2,  "Bob",   "US", False),
    (3,  "Chen",  "CN", True),
    (4,  "Diana", "US", None),
    (None, "Ghost","UK", False),     # NULL key to demo null join behavior
]

rows_orders = [
    (101, 1,   120.0, "IN"),
    (102, 1,    80.0, "IN"),
    (103, 2,    50.0, "US"),
    (104, 5,    30.0, "DE"),         # no matching customer_id
    (105, 3,   200.0, "CN"),
    (106, None, 15.0, "UK"),         # NULL key won’t match
    (107, 3,    40.0, "CN"),
    (108, 2,    75.0, "US"),
]

schema_customers = T.StructType([
    T.StructField("customer_id", T.IntegerType(), True),
    T.StructField("name",        T.StringType(),  True),
    T.StructField("country",     T.StringType(),  True),
    T.StructField("vip",         T.BooleanType(), True),
])

schema_orders = T.StructType([
    T.StructField("order_id",    T.IntegerType(), True),
    T.StructField("customer_id", T.IntegerType(), True),
    T.StructField("amount",      T.DoubleType(),  True),
    T.StructField("country",     T.StringType(),  True),  # same column name to show collisions
])

df_customers = spark.createDataFrame(rows_customers, schema_customers)
df_orders    = spark.createDataFrame(rows_orders,    schema_orders)

display(df_customers)
display(df_orders)

In [0]:
# INNER JOIN by default how="inner"
df_inner = df_orders.join(df_customers, on = "customer_id", how = "inner")
display(df_inner)

# Interview answer : An Inner Join returns only the records that have matching values in both tables based on the join condition. Non-matching rows from either table are excluded.


# Real-World Use Cases of Inner Join
# Use Case 1: Orders with Customer Details
# Scenario:
# Business wants to see customer information for each order.
# Tables:

# Orders
# Customers

# Why Inner Join?
# Only valid orders having a valid customer should appear in the report.

#----------------------------------------------------
# Use Case 2: Employee and Department Information
# Scenario:
# Generate a report containing:

# Employee Name
# Employee ID
# Department Name

# Tables:

# Employees
# Departments

# Why Inner Join?
# Only employees assigned to a department are required.

#----------------------------------------------------
# Use Case 3: Product Sales Reporting
# Scenario:
# Generate product sales reports.
# Tables:

# Sales
# Products

# Why Inner Join?
# Only sales having valid products should be included.

#----------------------------------------------------
# Use Case 5: Customer and Subscription Data
# Scenario:
# Find customers having active subscriptions.
# Tables:

# Customers
# Subscriptions

# Why Inner Join?
# Business wants only subscribed customers.

#----------------------------------------------------
# Use Case 7: ETL Data Enrichment
# Scenario:
# Enrich fact data with master data.
# Tables:

# Orders Fact Table
# Customer Dimension Table

# Why Inner Join?
# Only records having matching business keys should be processed.

#----------------------------------------------------
# I use Inner Join when I need only the matching records from two datasets. A common example is joining Orders and Customers tables to get customer details for valid orders. It is widely used in reporting, ETL pipelines, data enrichment, banking transactions, sales analytics, and employee management systems.

# Interview Keyword:
# "Used when only valid relationships between two tables are required." ✅

In [0]:
# df_orders is on left side
df_left = df_orders.join(df_customers, on = "customer_id", how = "left")
display(df_left)
# Left table = df_orders
# Right table = df_customers
# Because the syntax is left_dataframe.join(right_dataframe, ...)

# left(df_orders) ke sare rows aayege agar customer_id null ya (customer_id =5)not matching hai to left side null aayega
# df_customers is on left side
# df_right = df_orders.join(df_customers, on = "customer_id", how = "right")
# display(df_right)

# hai to right side null aayega

#-----------------------------------------------
# Interview Definition
# A Left Join (Left Outer Join) returns all records from the left table and the matching records from the right table. If there is no match in the right table, NULL values are returned for the right table columns.

#-----------------------------------------------
# Real-World Use Cases of Left Join
# Use Case 1: Orders and Customer Details
# Scenario:
# The business wants to see all orders, even if customer information is missing.
# Tables:

# Orders
# Customers

# Why Left Join?
# To ensure no order is lost in reporting.

#-----------------------------------------------
# Use Case 2: Employee and Manager Mapping
# Scenario:
# Show all employees along with their manager details.
# Tables:

# Employees
# Managers

# Why Left Join?
# Some employees may not have a manager assigned yet.

#-----------------------------------------------
# Use Case 3: Product Catalog and Sales
# Scenario:
# Business wants to see all products and their sales.
# Tables:

# Products
# Sales

# Why Left Join?
# Products with no sales should also appear in the report.

#-----------------------------------------------
# Use Case 4: Customer and Subscription Data
# Scenario:
# Generate a report of all customers and their subscriptions.
# Tables:

# Customers
# Subscriptions

# Why Left Join?
# Customers without subscriptions should also be visible.

# Interview Answer
# I use a Left Join when I want to retain all records from the left table and bring matching information from the right table. If a match is not found, the right-side columns contain NULL values. A common example is joining Orders with Customers where we want to see all orders, including those that have missing customer information.

In [0]:
# df_customers is on right side so all rows will be preserved
# df_orders is on left side
# Left table = df_orders
# Right table = df_customers
df_right = df_orders.join(df_customers, on = "customer_id", how = "right")
display(df_right)
# LEFT  = df_orders
# RIGHT = df_customers


# Interview Definition
# A Right Join (Right Outer Join) returns all records from the right table and the matching records from the left table. If a match is not found in the left table, NULL values are returned for the left table columns.

# I use a Right Join when I want to keep all records from the right table and bring matching data from the left table. If no match is found in the left table, the left-side columns are filled with NULL values.
#-----------------------------------------------

# Real-World Use Case
# Customers and Orders
# Scenario:
# Business wants to see all customers, including those who have never placed an order.
# Tables:

# Orders
# Customers

# Why Right Join?
# Because Customers is the primary dataset and no customer should be excluded.


#-----------------------------------------------
# One-Line Memory Trick
# Right Join = All Right Records + Matching Left Records
#-----------------------------------------------



In [0]:
#   Alias 
o = df_orders.alias("o")
o.show()

In [0]:
 o, c = df_orders.alias("o"), df_customers.alias("c")

 df_inner = o.join(c, on = "customer_id", how = "inner")
 display(df_inner) 

In [0]:
# giving alias to country column as they are in both the tables
df_inner_clean = (
    o.join(c, on = "customer_id", how="inner") \
    .select("order_id", "customer_id", "amount",
            F.col("o.country").alias("ship_country"),
            "name",
            F.col("c.country").alias("cust_country"),
            "vip"
            )
)
display(df_inner_clean)

In [0]:
 # Full join
 display(o.join(c, on='customer_id', how='full'))
 
# Interview Answer: What is a Full Join?

# A Full Join (Full Outer Join) returns all rows from both tables. If a match is found based on the join condition, the rows are combined; otherwise, NULL values are filled for the missing columns from the other table.

# When to use Full Outer Join
# I use Full Join when I need to compare two datasets completely and identify both matching and non-matching records from either side, such as data reconciliation, ETL validation, and audit reporting. ✅


#--------------------------------------------------------------
# Use Case 1: Customer vs Orders Analysis
# Scenario:
# A company wants to identify:

# Customers with orders
# Customers without orders
# Orders having an invalid customer
# Why Full Join?
# Because business wants to see complete coverage from both sides.


# Use Case 2: CRM vs ERP Data Reconciliation
# Scenario:
# Customer data exists in:

# CRM System
# ERP System

# Need to find:

# Customers present in both systems
# Customers missing in CRM
# Customers missing in ERP
# Why Full Join?
# Full Join helps compare both datasets at once.




# Use Case 3: Source vs Target Validation After ETL
# Scenario:
# Data is loaded from:

# Source Database
# Data Lake / Warehouse

# Need to verify:

# Missing records
# Extra records
# Successfully migrated records

# Why Full Join?
# Shows matched and unmatched records from both sides.



# Use Case 4: Orders vs Shipments
# Scenario:
# Business wants to check:

# Orders that are shipped
# Orders not shipped
# Shipments created without an order

# Tables:

# Orders
# Shipments

# Why Full Join?
# Provides complete picture of both business processes.


# Interview Answer

# I typically use a Full Join for data reconciliation and audit scenarios where I need to compare two datasets completely and identify matched records as well as records missing from either side. Examples include CRM vs ERP comparison, source-to-target ETL validation, bank reconciliation, and orders vs shipments analysis.




In [0]:
# left semi join
display(o.join(c, on="customer_id", how='left_semi')) # orders with a known customer

# 1. Sirf LEFT table ke columns aayenge.
# 2. Sirf woh rows aayengi jinka match RIGHT table me hai.

# LEFT SEMI JOIN = Left table se sirf wahi rows rakho
                #  jinka match Right table me milta hai.

# Notice: name column nahi aaya.​‌
# Kyuki left_semi sirf existence check karta hai.​‌
# Aur name column nahi aaya kyunki left_semi sirf filter karta hai, join karke right table ke columns nahi lata.


# left semi join me sirf left table hi aata but rows aati hai jinki match hoti hai right table se

# Interview Definition
# Left Semi Join returns only the rows from the left DataFrame that have at least one matching row in the right DataFrame. Only columns from the left DataFrame are returned.

# When to use
# Left Semi Join is used when we need only the rows from the left table that have a matching record in the right table, without bringing any columns from the right table. It is commonly used as an optimized form of SQL EXISTS queries.

# USE
# 1. Data Validation
# Check which orders have valid customers.​‌
# ✅ Sirf valid customer wale orders milenge.​‌

# 2. Filter Existing Records​‌
# Orders table se sirf wahi records chahiye
# jinke customer master table me present hain.​‌

# 3. ETL / Data Cleaning
# Invalid records hatane ke liye.​‌

# 4. Existence Check​‌
# "Is customer present?"​‌
# Bas existence check karna ho, customer details nahi chahiye.​‌




In [0]:
display(o.join(c, on="customer_id", how='left_anti')) # orders with a unknown customer

# "Left anti join wo hota hai jisme sirf left table ke columns aate hain aur rows wahi aati hain jinka match right table mein nahi milta."

# What is a Left Anti Join?
# A left anti join returns only those rows from the left table that do not have a match in the right table.

# "Left table ke wo rows jo right table me nahi milte."


# Interview Answer: What is Left Anti Join?
# Left Anti Join returns only those rows from the left DataFrame that do not have a matching row in the right DataFrame. It returns only the columns from the left DataFrame.

# USE:
# This is commonly used for:

# Data quality checks
# Finding orphan records
# Detecting invalid foreign keys
# Implementing SQL NOT EXISTS logic

In [0]:
# multiple keys to perform the join
df_multi = o.join(c, on=["customer_id", "country"], how='inner')
display(df_multi)

# dono columns match hone chahiye
# Yahaan 2 conditions hain:​‌
# customer_id match hona chahiye
# AND
# country bhi match hona chahiye
# Dono match honge tabhi row aayegi.

## SELF JOIN, CROSS JOIN, BROADCAST JOIN

In [0]:
from pyspark.sql import Row

emp = spark.createDataFrame([
    (1, "Asha", 3),
    (2, "Bob", 3),
    (3, "Chen", None),
    (4, "Diana", 2)
], ["emp_id", "emp_name", "manager_id"])

display(emp)


# There is no self join type in Spark.
# ✅ Self join means joining the same dataframe with aliases:

In [0]:

# Same dataframe ko khud se join karna.
# Hum manager ka naam nikalna chahte hain.
e = emp.alias("e")
m = emp.alias("m")

df_self = e.join(
    m,
    e.manager_id == m.emp_id,
    "left"
)

display(df_self)

# INTERVIEW ANSWER :A Self Join is a join where a table is joined with itself
# to find relationships between rows in the same table.

# Since both employees and managers are stored in the same table, we use a Self Join.


# when to use
# Employee -> Manager
# Category -> Parent Category
# Friend -> Friend
# Hierarchy Data

In [0]:
# 2. Cross Join
# Kya hai?
# Every row of left table joins with every row of right table.
teams = spark.createDataFrame([
    ("Team A",),
    ("Team B",)
], ["team"])

display(teams)

In [0]:
display(emp.crossJoin(teams))

# INTERVIEW ANSWER : Cross Join returns the Cartesian Product of two tables,
# meaning every row from the first table is combined with
# every row from the second table.


# When to Use?
# ✅ Generate all combinations

# 4 Employees × 2 Teams = 8 Rows
# When to use:
# Generate all combinations
# Testing
# Schedule generation
# Product × Region combinations

In [0]:
# 3. Broadcast Join ⭐ Interview Favorite

# INTERVIEW : Broadcast Join is a Spark optimization technique in which
# the small table is sent (broadcasted) to all worker nodes
# so Spark can join it with a large table without expensive shuffling.
# When to Use?
# ✅ One table is very small

# Q: Why is Broadcast Join faster?
# Because Spark broadcasts the small table to all executors,
# avoiding expensive data shuffling across the cluster.

# suppose :
# Orders = 100 Million rows
# Customers = 100 rows

# Agar normal join karoge to Spark data shuffle karega.
# Slow ❌


# Solution
# Small table ko broadcast karo.

from pyspark.sql.functions import broadcast

df = df_orders.join(
    broadcast(df_customers),
    "customer_id"
)

display(df)
# Visual
# Without Broadcast
# Worker 1 <--shuffle-->
# Worker 2 <--shuffle-->
# Worker 3 <--shuffle-->
# Bahut data movement.




# With Broadcast
# Small Customers Table

#           |
#      Broadcast
#           |

# Worker1
# Worker2
# Worker3

# Har worker ke paas customer table ki copy aa jati hai.
# No big shuffle
# Fast join


# Kab Use Karte Hain?
# Large Table + Small Table
# Fact + Dimension Join
# Orders + Customers
# Sales + Product Master
# Transactions + Country Master


## MEMORY TRICK FOR ALL JOIN
INNER JOIN     = Sirf Matching Rows

LEFT JOIN      = Keep All Left + Matching Right

RIGHT JOIN     = Keep All Right + Matching Left

FULL JOIN      = Keep Everything (Left + Right + Matches + Non-Matches)

LEFT SEMI      = Left Table Only + Matching Rows Only

LEFT ANTI      = Left Table Only + Non-Matching Rows Only

MULTIPLE KEYS  = Ek Nahi, Saare Keys Match Hone Chahiye

SELF JOIN      = Same Table Joins With Itself

CROSS JOIN     = Every Row × Every Row

BROADCAST JOIN = Small Table Ko Sab Workers Ko Bhej Do
                 Taaki Join Fast Ho Jaye

o, c = df_orders.alias("o"), df_customers.alias("c")

### Inner Join ✅
df_inner = o.join(c, on="customer_id", how="inner")

### Left Join ✅
df_left = o.join(c, on="customer_id", how="left")

### Right Join ✅
df_right = o.join(c, on="customer_id", how="right")

### Full Join ✅
df_full = o.join(c, on="customer_id", how="full")

### Left Semi Join ✅
df_left_semi = o.join(c, on="customer_id", how="left_semi")

### Left Anti Join ✅
df_left_anti = o.join(c, on="customer_id", how="left_anti")

### Multiple Keys Join ✅
df_multikeys = o.join(
    c,
    on=["customer_id", "name"],
    how="inner"
)

### Cross Join ✅
df_cross = o.crossJoin(c)

### Broadcast Join ✅
from pyspark.sql.functions import broadcast

df_broadcast = o.join(
    broadcast(c),
    on="customer_id",
    how="inner"
)

In [0]:
# 5. Self Join❌
# df_self = o.join(c, on='customer_id', how='self')
# There is no self join type in Spark.
# ✅ Self join means joining the same dataframe with aliases:
    
e = emp.alias("e")
m = emp.alias("m")

df_self = e.join(
    m,
    e.manager_id == m.emp_id,
    "left"
)

In [0]:
print("New joins")

# PySpark Joins Complete Notes

## Sample Tables

### Employee Table (`emp`)

```text
+--------+----------+---------+--------+
| emp_id | emp_name | dept_id | salary |
+--------+----------+---------+--------+
| 1      | Arif     | 10      | 50000  |
| 2      | John     | 20      | 60000  |
| 3      | Sara     | NULL    | 55000  |
| 4      | David    | 30      | 70000  |
| 5      | Mike     | 40      | 65000  |
| 6      | Emma     | NULL    | 62000  |
+--------+----------+---------+--------+
```

### Department Table (`dept`)

```text
+---------+--------------+
| dept_id | dept_name    |
+---------+--------------+
| 10      | HR           |
| 20      | Finance      |
| 30      | IT           |
| NULL    | Unknown Dept |
| 50      | Marketing    |
+---------+--------------+
```

---

# Relationship Diagram

```text

                dept_id
                    |
                    |
                    ▼

+----------------------+        +----------------------+
|      Employee        |        |      Department      |
+----------------------+        +----------------------+
| emp_id               |        | dept_id (PK)         |
| emp_name             |------->| dept_name            |
| dept_id (FK)         |        +----------------------+
| salary               |
+----------------------+

```

---

# 1. INNER JOIN

## Input Tables

### Employee

```text
+--------+----------+---------+
| emp_id | emp_name | dept_id |
+--------+----------+---------+
| 1      | Arif     | 10      |
| 2      | John     | 20      |
| 3      | Sara     | NULL    |
| 4      | David    | 30      |
| 5      | Mike     | 40      |
| 6      | Emma     | NULL    |
+--------+----------+---------+
```

### Department

```text
+---------+--------------+
| dept_id | dept_name    |
+---------+--------------+
| 10      | HR           |
| 20      | Finance      |
| 30      | IT           |
| NULL    | Unknown Dept |
| 50      | Marketing    |
+---------+--------------+
```

## Query

```python
emp.join(
    dept,
    emp.dept_id == dept.dept_id,
    "inner"
)
```

## Output

```text
+--------+----------+---------+--------+-----------+
| emp_id | emp_name | dept_id | salary | dept_name |
+--------+----------+---------+--------+-----------+
| 1      | Arif     | 10      | 50000  | HR        |
| 2      | John     | 20      | 60000  | Finance   |
| 4      | David    | 30      | 70000  | IT        |
+--------+----------+---------+--------+-----------+
```

### Explanation

Only matching records from both tables are returned.

---

# 2. LEFT JOIN

## Input Tables

### Employee

```text
+--------+----------+---------+
| emp_id | emp_name | dept_id |
+--------+----------+---------+
| 1      | Arif     | 10      |
| 2      | John     | 20      |
| 3      | Sara     | NULL    |
| 4      | David    | 30      |
| 5      | Mike     | 40      |
| 6      | Emma     | NULL    |
+--------+----------+---------+
```

### Department

```text
+---------+--------------+
| dept_id | dept_name    |
+---------+--------------+
| 10      | HR           |
| 20      | Finance      |
| 30      | IT           |
| NULL    | Unknown Dept |
| 50      | Marketing    |
+---------+--------------+
```

## Query

```python
emp.join(
    dept,
    emp.dept_id == dept.dept_id,
    "left"
)
```

## Output

```text
+--------+----------+---------+--------+-----------+
| emp_id | emp_name | dept_id | salary | dept_name |
+--------+----------+---------+--------+-----------+
| 1      | Arif     | 10      | 50000  | HR        |
| 2      | John     | 20      | 60000  | Finance   |
| 3      | Sara     | NULL    | 55000  | NULL      |
| 4      | David    | 30      | 70000  | IT        |
| 5      | Mike     | 40      | 65000  | NULL      |
| 6      | Emma     | NULL    | 62000  | NULL      |
+--------+----------+---------+--------+-----------+
```

### Explanation

Returns all records from the left table.

---

# 3. RIGHT JOIN

## Input Tables

### Employee

```text
+--------+----------+---------+
| emp_id | emp_name | dept_id |
+--------+----------+---------+
| 1      | Arif     | 10      |
| 2      | John     | 20      |
| 3      | Sara     | NULL    |
| 4      | David    | 30      |
| 5      | Mike     | 40      |
| 6      | Emma     | NULL    |
+--------+----------+---------+
```

### Department

```text
+---------+--------------+
| dept_id | dept_name    |
+---------+--------------+
| 10      | HR           |
| 20      | Finance      |
| 30      | IT           |
| NULL    | Unknown Dept |
| 50      | Marketing    |
+---------+--------------+
```

## Query

```python
emp.join(
    dept,
    emp.dept_id == dept.dept_id,
    "right"
)
```

## Output

```text
+--------+----------+---------+--------+--------------+
| emp_id | emp_name | dept_id | salary | dept_name    |
+--------+----------+---------+--------+--------------+
| 1      | Arif     | 10      | 50000  | HR           |
| 2      | John     | 20      | 60000  | Finance      |
| 4      | David    | 30      | 70000  | IT           |
| NULL   | NULL     | NULL    | NULL   | Unknown Dept |
| NULL   | NULL     | 50      | NULL   | Marketing    |
+--------+----------+---------+--------+--------------+
```

### Explanation

Returns all records from the right table.

---

# 4. FULL OUTER JOIN

## Query

```python
emp.join(
    dept,
    emp.dept_id == dept.dept_id,
    "outer"
)
```

## Output

```text
+--------+----------+---------+--------+--------------+
| emp_id | emp_name | dept_id | salary | dept_name    |
+--------+----------+---------+--------+--------------+
| 1      | Arif     | 10      | 50000  | HR           |
| 2      | John     | 20      | 60000  | Finance      |
| 4      | David    | 30      | 70000  | IT           |
| 3      | Sara     | NULL    | 55000  | NULL         |
| 6      | Emma     | NULL    | 62000  | NULL         |
| 5      | Mike     | 40      | 65000  | NULL         |
| NULL   | NULL     | NULL    | NULL   | Unknown Dept |
| NULL   | NULL     | 50      | NULL   | Marketing    |
+--------+----------+---------+--------+--------------+
```

### Explanation

Returns all matching and non-matching rows from both tables.

---

# 5. LEFT SEMI JOIN

## Query

```python
emp.join(
    dept,
    emp.dept_id == dept.dept_id,
    "left_semi"
)
```

## Output

```text
+--------+----------+---------+--------+
| emp_id | emp_name | dept_id | salary |
+--------+----------+---------+--------+
| 1      | Arif     | 10      | 50000  |
| 2      | John     | 20      | 60000  |
| 4      | David    | 30      | 70000  |
+--------+----------+---------+--------+
```

### Explanation

Returns only left table columns where a match exists.

---

# 6. LEFT ANTI JOIN

## Query

```python
emp.join(
    dept,
    emp.dept_id == dept.dept_id,
    "left_anti"
)
```

## Output

```text
+--------+----------+---------+--------+
| emp_id | emp_name | dept_id | salary |
+--------+----------+---------+--------+
| 3      | Sara     | NULL    | 55000  |
| 5      | Mike     | 40      | 65000  |
| 6      | Emma     | NULL    | 62000  |
+--------+----------+---------+--------+
```

### Explanation

Returns rows from the left table that do not have a matching row in the right table.

---

# 7. CROSS JOIN

## Query

```python
emp.crossJoin(dept)
```

## Output Count

```text
Employee Records   = 6
Department Records = 5

6 × 5 = 30 rows
```

### Explanation

Every employee is joined with every department.

---

# 8. SELF JOIN

## Sample Employee Table

```text
+--------+----------+--------+
| emp_id | emp_name | mgr_id |
+--------+----------+--------+
| 1      | Arif     | NULL   |
| 2      | John     | 1      |
| 3      | Sara     | 1      |
| 4      | David    | 2      |
+--------+----------+--------+
```

## Query

```python
e = emp.alias("e")
m = emp.alias("m")

e.join(
    m,
    e.mgr_id == m.emp_id,
    "left"
)
```

## Output

```text
+----------+---------+
| Employee | Manager |
+----------+---------+
| John     | Arif    |
| Sara     | Arif    |
| David    | John    |
+----------+---------+
```

---

# NULL Handling In Joins

## Important

```python
emp.dept_id == dept.dept_id
```

```text
NULL = NULL

FALSE
```

NULL never matches NULL.

## Null Safe Join

```python
emp.join(
    dept,
    emp.dept_id.eqNullSafe(dept.dept_id),
    "inner"
)
```

## Output

```text
+----------+--------------+
| emp_name | dept_name    |
+----------+--------------+
| Arif     | HR           |
| John     | Finance      |
| David    | IT           |
| Sara     | Unknown Dept |
| Emma     | Unknown Dept |
+----------+--------------+
```

---

# Most Asked Interview Questions

### Q1. Difference between INNER JOIN and LEFT JOIN?

**Answer:**  
INNER JOIN returns only matched rows.  
LEFT JOIN returns all rows from left table and matched rows from right table.

### Q2. What is LEFT SEMI JOIN?

**Answer:**  
Returns only columns from the left table where matching records exist.

### Q3. What is LEFT ANTI JOIN?

**Answer:**  
Returns records from the left table that do not have matching records in the right table.

### Q4. Does NULL equal NULL in Spark Join?

**Answer:** No.

```python
NULL = NULL -> FALSE
```

Use:

```python
eqNullSafe()
```

### Q5. What is Broadcast Join?

```python
from pyspark.sql.functions import broadcast

emp.join(
    broadcast(dept),
    "dept_id"
)
```

Small table is pushed to executors to avoid shuffle.

### Q6. What is Shuffle?

Data movement across executors during joins.

### Q7. Which joins are most expensive?

- Full Outer Join
- Cross Join

### Q8. Default Join Strategy In Spark?

**Sort Merge Join**

### Q9. How To Improve Join Performance?

1. Broadcast Join
2. Filter Early
3. Select Required Columns
4. Partition Data Correctly
5. Use Bucketing

### Q10. What is `eqNullSafe()`?

Used to match NULL values during joins.

---

# Quick Revision

```text
INNER      -> Matching Rows

LEFT       -> All Left + Match

RIGHT      -> All Right + Match

FULL       -> Everything

LEFT SEMI  -> Exists

LEFT ANTI  -> Not Exists

CROSS      -> Cartesian Product

SELF       -> Table With Itself

NULL != NULL

eqNullSafe() -> NULL Matching

Broadcast Join -> Reduces Shuffle

Sort Merge Join -> Default For Large Datasets
```